# BahuSanket · 20 · Live demo: type a paragraph, the avatar signs it

This notebook is for **presenting**. It starts a small web server on this Colab GPU and gives you a link to the **BahuSanket front end**:
a text box, a language picker, the avatar and live captions. Type (or paste) English, Hindi or Kannada, press **Sign it**, and a few
seconds later the avatar signs it. The six prepared paragraphs from notebook 19 are already in its "Signed so far" list.
Behind the button is exactly the pipeline of notebook 19, nothing new is trained:

IndicTrans2 (Hindi/Kannada → English) → split into sentences → our text model (guidance s = 4, length control, extra codes, bone fix)
→ sentences blended into one sequence (ease 8, chosen in notebook 19) → depth model (notebook 18) → avatar.

**On presentation day**
1. About **15 minutes before**: Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all** (about 5–8 minutes).
2. The last cell prints **a link**. Open it (it opens in a new tab, in this same browser). Try one sentence once, so everything is warm.
3. **Keep this Colab tab open** during the presentation (Colab stops idle sessions after a while; a tab left open is usually enough).
4. Present from that tab (F11 makes it full screen). Ctrl+Enter also presses Sign it; everything you sign stays in "Signed so far".
   The research viewer with all the switches is at the same link + `viewer.html`.

**Backups, in case the Wi-Fi or Colab fails:** the published avatar page (https://claude.ai/artifact/6AKrgZDSC7SeuSdcXVWFgb) already has
the six demo paragraphs in English, Hindi and Kannada, and a short screen recording of this live demo is worth making the day before.

The Hugging Face token is read from Colab **Secrets** (`HF_TOKEN`), or a hidden box asks for it. It is never printed or saved.

In [ ]:
# 1. Setup (+ the HF token, needed for IndicTrans2)
import os, sys, json, glob, time, subprocess, math
if os.environ.get("BAHU_ROOT"):                     # (used only for Claude's local test run)
    ROOT = os.environ["BAHU_ROOT"]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/BahuSanket"
    if not os.path.isdir(ROOT):
        hits = glob.glob("/content/drive/MyDrive/**/BahuSanket", recursive=True)
        assert hits, "Could not find the BahuSanket folder in your Drive"
        ROOT = hits[0]
print("Drive folder:", ROOT)
for mod, pkg in (("sentence_transformers", "sentence-transformers"), ("sacrebleu", "sacrebleu")):
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
CODE_DIR = os.path.join(ROOT, "code"); sys.path.insert(0, CODE_DIR)
RESULTS_DIR = os.path.join(ROOT, "results"); os.makedirs(RESULTS_DIR, exist_ok=True)
import importlib, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import bahusanket_lib as B; importlib.reload(B)
import bahusanket_vq as V; importlib.reload(V)
import bahusanket_t2c as T; importlib.reload(T)
import bahusanket_layout as L; importlib.reload(L)
import bahusanket_viz as VZ; importlib.reload(VZ)
LAY = L.apply(json.load(open(os.path.join(ROOT, "model_point_indices_v2.json"))), B, V)
POINT_IDX = json.load(open(os.path.join(ROOT, "model_point_indices_v2.json")))
VIZ_DIR = os.path.join(ROOT, "viz"); os.makedirs(VIZ_DIR, exist_ok=True)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
QUICK = bool(os.environ.get("BAHU_QUICK"))
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: this will be very slow)", "|", B.N_POINTS, "points")
if not os.environ.get("HF_TOKEN") and not os.environ.get("BAHU_ROOT"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        # Secrets only work from the Colab web page (not e.g. VS Code). Ask instead: typed in a hidden box, never printed or saved.
        from getpass import getpass
        os.environ["HF_TOKEN"] = getpass("Colab Secrets not available. Paste your Hugging Face token (hidden) and press Enter: ").strip()
    assert os.environ.get("HF_TOKEN", "").startswith("hf_"), "That does not look like a Hugging Face token (it starts with hf_)"
    print("Hugging Face token: OK (not shown)")

In [ ]:
# 2. Module code: notebook 13's vocabulary (bahusanket_rvq.py) and the extra-code predictor (bahusanket_rtrans.py)
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import bahusanket_lib as B
import bahusanket_vq as V


class EMAQuantizer(nn.Module):
    """One codebook updated by moving average with code reset (same recipe as the base vocabulary)."""
    def __init__(self, n_codes, dim, mu=0.99):
        super().__init__()
        self.n_codes, self.dim, self.mu = n_codes, dim, mu
        self.register_buffer("codebook", torch.zeros(n_codes, dim))
        self.register_buffer("code_sum", torch.zeros(n_codes, dim))
        self.register_buffer("code_count", torch.ones(n_codes))
        self.register_buffer("initted", torch.tensor(False))

    @staticmethod
    def _random_rows(z, n):
        if len(z) < n:
            z = z.repeat((n + len(z) - 1) // len(z), 1)
            z = z + torch.randn_like(z) * 0.01 / np.sqrt(z.shape[1])
        return z[torch.randperm(len(z), device=z.device)[:n]]

    def forward(self, z):
        """z (N, dim) float32 -> (idx (N,), quantised (N, dim), stats)"""
        if self.training and not bool(self.initted):
            init = self._random_rows(z.detach(), self.n_codes)
            self.codebook.copy_(init); self.code_sum.copy_(init); self.code_count.fill_(1.0)
            self.initted.fill_(True)
        d = z.pow(2).sum(1, keepdim=True) - 2 * z @ self.codebook.t() + self.codebook.pow(2).sum(1)[None]
        idx = d.argmin(1)
        if self.training:
            with torch.no_grad():
                onehot = F.one_hot(idx, self.n_codes).type(z.dtype)
                cnt, s = onehot.sum(0), onehot.t() @ z.detach()
                self.code_sum.mul_(self.mu).add_(s, alpha=1 - self.mu)
                self.code_count.mul_(self.mu).add_(cnt, alpha=1 - self.mu)
                used = (self.code_count >= 1.0).float()[:, None]
                fresh = self._random_rows(z.detach(), self.n_codes)
                self.codebook.copy_(used * self.code_sum / self.code_count[:, None].clamp(min=1e-5) + (1 - used) * fresh)
                self.code_sum.copy_(used * self.code_sum + (1 - used) * fresh)
                self.code_count.copy_(used[:, 0] * self.code_count + (1 - used[:, 0]))
        prob = torch.bincount(idx, minlength=self.n_codes).float() / len(idx)
        perp = torch.exp(-(prob * torch.log(prob + 1e-10)).sum())
        return idx, self.codebook[idx], {"perplexity": perp, "used": (prob > 0).float().mean()}


class ResidualPoseVQ(nn.Module):
    def __init__(self, base, n_res=3, n_codes=512):
        super().__init__()
        self.base = base.eval()
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.decoder = copy.deepcopy(base.decoder)            # starts as the old decoder; trained
        for p in self.decoder.parameters():
            p.requires_grad_(True)
        self.n_res = n_res
        self.res = nn.ModuleList([EMAQuantizer(n_codes, base.code_dim) for _ in range(n_res)])

    def train(self, mode=True):
        super().train(mode)
        self.base.eval()                                      # the base vocabulary never changes
        return self

    def trainable_parameters(self):
        return list(self.decoder.parameters())

    @torch.no_grad()
    def _latents(self, x):
        z = self.base.encoder(x.transpose(1, 2))              # (B, C, Tq)
        Bn, C, Tq = z.shape
        zf = z.transpose(1, 2).reshape(-1, C).float()
        with torch.autocast(device_type=zf.device.type, enabled=False):
            idx0, zq0, _ = self.base.quantize(zf)             # base is in eval mode: no codebook change
        return zf, idx0, zq0, Bn, Tq

    def forward(self, x, keep=None):
        """x (B, T, IN_DIM). keep: (B,) number of residual layers kept per clip (None = all). Returns recon, stats, codes."""
        zf, idx0, zq0, Bn, Tq = self._latents(x)
        r, total, stats, codes = zf - zq0, zq0.clone(), {}, [idx0]
        with torch.autocast(device_type=zf.device.type, enabled=False):
            for l, qz in enumerate(self.res):
                idx, q, st = qz(r)
                r = r - q
                gate = torch.ones(len(zf), 1, device=zf.device) if keep is None else \
                    (l < keep).float().repeat_interleave(Tq)[:, None]
                total = total + q * gate
                codes.append(idx)
                stats[f"perp{l + 1}"], stats[f"used{l + 1}"] = st["perplexity"], st["used"]
                stats[f"res{l + 1}"] = r.pow(2).mean()        # how much is still unexplained after layer l+1
        stats["res0"] = (zf - zq0).pow(2).mean()
        out = self.decoder(total.view(Bn, Tq, -1).transpose(1, 2)).transpose(1, 2)
        return out, stats, torch.stack([c.view(Bn, Tq) for c in codes], -1)

    @torch.no_grad()
    def encode(self, x):
        """-> codes (B, T/4, 1 + n_res); column 0 is the unchanged base code."""
        was = self.training
        self.eval()
        _, _, codes = self.forward(x)
        self.train(was)
        return codes

    @torch.no_grad()
    def decode(self, codes, n_res=None):
        """codes (B, Tq, 1 + n_res) -> poses (B, T, IN_DIM), using the base code plus the first n_res residual codes."""
        n_res = self.n_res if n_res is None else n_res
        zq = self.base.codebook[codes[..., 0]]
        for l in range(n_res):
            zq = zq + self.res[l].codebook[codes[..., l + 1]]
        return self.decoder(zq.transpose(1, 2)).transpose(1, 2)


@torch.no_grad()
def tokenize_and_rebuild_rvq(model, split, idx, device, batch_size=64, n_res_list=None):
    """Whole clips -> codes (list of (nq, 1+n_res) int16) and rebuilt poses for each n_res in n_res_list."""
    model.eval()
    n_res_list = [model.n_res] if n_res_list is None else n_res_list
    codes_out, rec_out = [], {n: [] for n in n_res_list}
    for s in range(0, len(idx), batch_size):
        chunk = idx[s:s + batch_size]
        Ls = [len(split["poses"][i]) for i in chunk]
        T = int(np.ceil(max(Ls) / V.DOWN) * V.DOWN)
        x = np.zeros((len(chunk), T, V.IN_DIM), np.float32)
        for k, i in enumerate(chunk):
            p = split["poses"][i][..., :2].astype(np.float32).copy()
            p[~split["masks"][i]] = 0.0
            x[k, :Ls[k]] = p.reshape(Ls[k], -1)
        codes = model.encode(torch.from_numpy(x).to(device))
        recs = {n: model.decode(codes, n).float().cpu().numpy().reshape(len(chunk), T, B.N_POINTS, 2) for n in n_res_list}
        codes = codes.cpu().numpy()
        for k, L in enumerate(Ls):
            nq = int(np.ceil(L / V.DOWN))
            codes_out.append(codes[k, :nq].astype(np.int16))
            for n in n_res_list:
                rec_out[n].append(recs[n][k, :L])
    return codes_out, rec_out


@torch.no_grad()
def codes_to_poses_rvq(model, code_list, device, n_res=None, batch_size=64):
    """List of (nq, 1+n_res) code arrays -> list of (T, N_POINTS, 2) poses."""
    model.eval(); out = []
    for s in range(0, len(code_list), batch_size):
        ch = code_list[s:s + batch_size]
        nq = max(len(c) for c in ch)
        arr = np.zeros((len(ch), nq, ch[0].shape[1]), np.int64)
        for k, c in enumerate(ch):
            arr[k, :len(c)] = c
        rec = model.decode(torch.from_numpy(arr).to(device), n_res).float().cpu().numpy().reshape(len(ch), nq * V.DOWN, B.N_POINTS, 2)
        out += [rec[k, :len(c) * V.DOWN] for k, c in enumerate(ch)]
    return out


import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import bahusanket_lib as B


class ResidualCodePredictor(nn.Module):
    def __init__(self, n_layers=4, n_codes=512, d_model=384, n_heads=6, n_blocks=4, ff=1536, dropout=0.1,
                 max_len=100, text_encoder="distilbert", vocab_size=30522):
        super().__init__()
        self.n_layers, self.n_codes, self.max_len = n_layers, n_codes, max_len
        self.PAD = n_codes
        self.text_encoder = text_encoder
        if text_encoder == "distilbert":
            from transformers import AutoModel
            self.bert = AutoModel.from_pretrained("distilbert-base-uncased")
            self.bert.requires_grad_(False); self.bert.eval()
            self.txt_in = nn.Linear(self.bert.config.dim, d_model)
        else:                                       # (small local tests only)
            self.tok = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.txt_pos = nn.Embedding(B.MAX_TOKENS, d_model)
        self.code_emb = nn.ModuleList([nn.Embedding(n_codes + 1, d_model, padding_idx=n_codes) for _ in range(n_layers - 1)])
        self.layer_emb = nn.Embedding(n_layers, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        self.blocks = nn.TransformerDecoder(nn.TransformerDecoderLayer(
            d_model, n_heads, ff, dropout, batch_first=True, norm_first=True), n_blocks)
        self.norm = nn.LayerNorm(d_model)
        self.heads = nn.ModuleList([nn.Linear(d_model, n_codes) for _ in range(n_layers - 1)])   # one per extra layer
        self.drop = nn.Dropout(dropout)

    def train(self, mode=True):
        super().train(mode)
        if self.text_encoder == "distilbert":
            self.bert.eval()
        return self

    def encode_text(self, ids, attn):
        if self.text_encoder == "distilbert":
            with torch.no_grad():
                h = self.bert(input_ids=ids, attention_mask=attn).last_hidden_state
            h = self.txt_in(h.float())
        else:
            h = self.tok(ids)
        return self.drop(h + self.txt_pos(torch.arange(ids.shape[1], device=ids.device))), attn == 0

    def forward(self, codes, j, mem, mem_pad, pad):
        """codes (B, T, n_layers) with PAD where missing; j (B,) target layer in 1..n_layers-1 -> logits (B, T, n_codes)."""
        Bn, T, _ = codes.shape
        h = self.pos(torch.arange(T, device=codes.device))[None] + self.layer_emb(j)[:, None]
        for l in range(self.n_layers - 1):          # sum of the embeddings of the layers below j
            use = (l < j).float()[:, None, None]
            h = h + self.code_emb[l](codes[..., l]) * use
        h = self.blocks(self.drop(h), mem, tgt_key_padding_mask=pad, memory_key_padding_mask=mem_pad)
        h = self.norm(h)
        out = torch.stack([hd(h) for hd in self.heads], 1)                       # (B, n_layers-1, T, n_codes)
        return out[torch.arange(Bn, device=codes.device), j - 1]

    @torch.no_grad()
    def fill(self, ids, attn, base, temperature=0.0):
        """base: list of 1-D base-code arrays -> list of (len, n_layers) arrays. temperature 0 = most likely code."""
        n, T = len(base), min(self.max_len, max(len(b) for b in base))
        dev = ids.device
        codes = torch.full((n, T, self.n_layers), self.PAD, dtype=torch.long, device=dev)
        for k, b in enumerate(base):
            codes[k, :min(len(b), T), 0] = torch.as_tensor(np.asarray(b[:T], np.int64), device=dev)
        pad = codes[..., 0] == self.PAD
        mem, mem_pad = self.encode_text(ids, attn)
        for j in range(1, self.n_layers):
            lg = self.forward(codes, torch.full((n,), j, dtype=torch.long, device=dev), mem, mem_pad, pad).float()
            if temperature > 0:
                nxt = torch.multinomial(F.softmax(lg / temperature, -1).view(-1, self.n_codes), 1).view(n, T)
            else:
                nxt = lg.argmax(-1)
            codes[..., j] = torch.where(pad, torch.full_like(nxt, self.PAD), nxt)
        codes = codes.cpu().numpy()
        out = []
        for k, b in enumerate(base):
            c = codes[k, :min(len(b), T)]
            if len(b) > T:                           # (longer than the model can see: repeat the last extras)
                c = np.concatenate([c, np.concatenate([np.asarray(b[T:], np.int64)[:, None],
                                                       np.repeat(c[-1:, 1:], len(b) - T, 0)], 1)])
            out.append(c.astype(np.int64))
        return out


def residual_batch(codes_list, n_layers, pad_id, max_len, corrupt=0.0, j_fixed=None):
    """Batch of (len, n_layers) code arrays -> codes (B, T, L) with PAD, target layer j (B,), targets (B, T) (-100 = ignore)."""
    T = min(max_len, max(len(c) for c in codes_list))
    n = len(codes_list)
    x = np.full((n, T, n_layers), pad_id, np.int64)
    j = np.random.randint(1, n_layers, n) if j_fixed is None else np.full(n, j_fixed)
    tgt = np.full((n, T), -100, np.int64)
    for k, c in enumerate(codes_list):
        c = np.asarray(c, np.int64)[:T]
        x[k, :len(c)] = c
        tgt[k, :len(c)] = c[:, j[k]]
        if corrupt > 0 and len(c):                   # replace a few input codes with random ones (robustness to generated base codes)
            hit = np.random.rand(len(c), n_layers) < np.random.uniform(0, corrupt)
            hit[:, j[k]:] = False
            x[k, :len(c)][hit] = np.random.randint(0, pad_id, hit.sum())
    t = torch.as_tensor
    return t(x), t(j), t(tgt)


@torch.no_grad()
def fill_soft(m, rvq, ids, attn, base, temperature=1.0):
    """Like ResidualCodePredictor.fill, but returns BLENDED latents: base vector + for each extra layer the probability-weighted
    average of that layer's code vectors. The most likely code is still used to condition the next layer."""
    n, T = len(base), min(m.max_len, max(len(b) for b in base))
    dev = ids.device
    codes = torch.full((n, T, m.n_layers), m.PAD, dtype=torch.long, device=dev)
    for k, b in enumerate(base):
        codes[k, :min(len(b), T), 0] = torch.as_tensor(np.asarray(b[:T], np.int64), device=dev)
    pad = codes[..., 0] == m.PAD
    mem, mem_pad = m.encode_text(ids, attn)
    z = rvq.base.codebook[codes[..., 0].clamp(max=rvq.base.n_codes - 1)]
    for j in range(1, m.n_layers):
        lg = m.forward(codes, torch.full((n,), j, dtype=torch.long, device=dev), mem, mem_pad, pad).float()
        z = z + F.softmax(lg / temperature, -1) @ rvq.res[j - 1].codebook
        codes[..., j] = torch.where(pad, torch.full_like(pad, m.PAD, dtype=torch.long), lg.argmax(-1))
    z = z.cpu().numpy()
    return [z[k, :min(len(b), T)] for k, b in enumerate(base)]

@torch.no_grad()
def latents_to_poses(rvq, lat_list, device, batch_size=64):
    out = []
    for s in range(0, len(lat_list), batch_size):
        ch = lat_list[s:s + batch_size]; nq = max(len(c) for c in ch)
        arr = np.zeros((len(ch), nq, ch[0].shape[1]), np.float32)
        for k, c in enumerate(ch):
            arr[k, :len(c)] = c
        rec = rvq.decoder(torch.from_numpy(arr).to(device).transpose(1, 2)).transpose(1, 2).float().cpu().numpy()
        rec = rec.reshape(len(ch), nq * V.DOWN, B.N_POINTS, 2)
        out += [rec[k, :len(c) * V.DOWN] for k, c in enumerate(ch)]
    return out

In [ ]:
# 3. Load notebook 13's vocabulary and its codes
VQ_RUN = f"vq_v2_{B.N_POINTS}pts"; RV_RUN = f"rvq_v2_{B.N_POINTS}pts"
ck_vq = torch.load(os.path.join(ROOT, "checkpoints", VQ_RUN, "last.pt"), map_location=DEVICE, weights_only=False); vc = ck_vq["config"]
vq = V.PoseVQVAE(in_dim=B.N_POINTS * 2, width=vc["width"], code_dim=vc["code_dim"], n_codes=vc["n_codes"]).to(DEVICE)
vq.load_state_dict(ck_vq["model"]); vq.eval()
ck_rv = torch.load(os.path.join(ROOT, "checkpoints", RV_RUN, "last.pt"), map_location=DEVICE, weights_only=False); RVC = ck_rv["config"]
rvq = ResidualPoseVQ(vq, n_res=RVC["n_res"], n_codes=RVC["n_codes"]).to(DEVICE)
rvq.load_state_dict(ck_rv["model"]); rvq.eval()
N_LAYERS = 1 + RVC["n_res"]
print(f"vocabulary {RV_RUN}: step {ck_rv['step']}, base + {RVC['n_res']} extra layers")
TR = os.path.join(ROOT, "training_ready_v2")
val = B.load_split(os.path.join(TR, "val.npz"), verbose=False)
test = B.load_split(os.path.join(TR, "test.npz"), verbose=False)
rtok = np.load(os.path.join(ROOT, "tokens", f"pose_codes_{RV_RUN}.npz"), allow_pickle=True)
for name, d in [("val", val), ("test", test)]:
    assert list(rtok[f"{name}_uids"]) == list(d["uids"]), f"{name}: code file and pose file disagree"
codes = {s: list(rtok[f"{s}_codes"]) for s in ["train", "val", "test"]}
texts = {s: [str(t) for t in rtok[f"{s}_texts"]] for s in ["train", "val", "test"]}
print({s: len(codes[s]) for s in codes})

In [ ]:
# 4. The extra-code predictor from notebook 14b (rtrans_123pts_b, best pass)
ckp = torch.load(os.path.join(ROOT, "checkpoints", f"rtrans_{B.N_POINTS}pts_b", "best.pt"), map_location=DEVICE, weights_only=False)
PR = ckp["config"]
pred = ResidualCodePredictor(n_layers=N_LAYERS, n_codes=RVC["n_codes"], d_model=PR["d_model"], n_heads=PR["n_heads"],
                             n_blocks=PR["n_blocks"], ff=PR["ff"], dropout=PR["dropout"], max_len=PR["max_len"]).to(DEVICE)
pred.load_state_dict(ckp["model"]); pred.eval()
print("extra-code predictor: pass", ckp["epoch"])

In [ ]:
# 5. Load notebook 10's matcher (also gives the MiniLM sentence encoder)
from sentence_transformers import SentenceTransformer
from scipy import linalg
STRIDE, MAXT = 2, 150
mck = torch.load(os.path.join(ROOT, "checkpoints", "matcher_v1", "best.pt"), map_location=DEVICE, weights_only=False)
FMEAN, FSTD = mck["fmean"], mck["fstd"]

def clip_feats(p):
    p = np.asarray(p, np.float32)[..., :2][::STRIDE][:MAXT]
    m = np.abs(p).sum(-1) > 0
    v = np.zeros_like(p); v[1:] = p[1:] - p[:-1]
    v[1:][~(m[1:] & m[:-1])] = 0
    return np.concatenate([p.reshape(len(p), -1), v.reshape(len(p), -1)], 1)

def batch_feats(seqs):
    fs = [(clip_feats(s) - FMEAN) / FSTD for s in seqs]
    Tm = max(1, max(len(f) for f in fs))
    x = np.zeros((len(fs), Tm, fs[0].shape[1]), np.float32); pad = np.ones((len(fs), Tm), bool)
    for k, f in enumerate(fs):
        x[k, :len(f)] = f; pad[k, :len(f)] = False
    return torch.as_tensor(x, device=DEVICE), torch.as_tensor(pad, device=DEVICE)

class MotionEnc(nn.Module):
    def __init__(self, in_dim, d=256, layers=4, out=256):
        super().__init__()
        self.inp = nn.Linear(in_dim, d); self.pos = nn.Embedding(MAXT, d)
        self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(d, 4, 4 * d, 0.1, batch_first=True, norm_first=True), layers)
        self.out = nn.Linear(d, out)
    def forward(self, x, pad):
        h = self.inp(x) + self.pos(torch.arange(x.shape[1], device=x.device))
        h = self.enc(h, src_key_padding_mask=pad)
        keep = (~pad).float()[..., None]
        return F.normalize(self.out((h * keep).sum(1) / keep.sum(1).clamp(min=1)), dim=-1)

class TextEnc(nn.Module):
    def __init__(self, in_dim=384, out=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, 512), nn.GELU(), nn.Dropout(0.1), nn.Linear(512, out))
    def forward(self, e):
        return F.normalize(self.net(e), dim=-1)

menc, tenc = MotionEnc(2 * B.N_POINTS * 2).to(DEVICE), TextEnc().to(DEVICE)
menc.load_state_dict(mck["menc"]); tenc.load_state_dict(mck["tenc"]); menc.eval(); tenc.eval()
st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

@torch.no_grad()
def embed_motion(seqs, bs=256):
    return np.concatenate([menc(*batch_feats(seqs[s:s + bs])).float().cpu().numpy() for s in range(0, len(seqs), bs)])
@torch.no_grad()
def embed_sentences(tx):
    e = st.encode(list(tx), batch_size=512, convert_to_numpy=True, show_progress_bar=False).astype(np.float32)
    return tenc(torch.as_tensor(e, device=DEVICE)).float().cpu().numpy()
def r_precision(me, te, pool=32, seed=0):
    n = (len(me) // pool) * pool
    order = np.random.RandomState(seed).permutation(len(me))[:n]
    hits = np.zeros(3)
    for s in range(0, n, pool):
        ix = order[s:s + pool]; sim = me[ix] @ te[ix].T
        rank = (sim > np.diag(sim)[:, None]).sum(1)
        hits += [(rank < k).sum() for k in (1, 2, 3)]
    return hits / max(n, 1)
def fid(a, b):
    mu1, mu2 = a.mean(0), b.mean(0); s1, s2 = np.cov(a, rowvar=False), np.cov(b, rowvar=False)
    cs = linalg.sqrtm(s1 @ s2); cs = cs[0] if isinstance(cs, tuple) else cs
    return float(((mu1 - mu2) ** 2).sum() + np.trace(s1 + s2 - 2 * np.real(cs)))
def diversity(e, n=300, seed=0):
    rs = np.random.RandomState(seed); a, b = rs.randint(0, len(e), n), rs.randint(0, len(e), n)
    return float(np.linalg.norm(e[a] - e[b], axis=1).mean())
def scores(seqs, te, real_e):
    me = embed_motion(seqs); r = r_precision(me, te)
    return {"top1": r[0], "top2": r[1], "top3": r[2], "fid": fid(me, real_e), "div": diversity(me),
            "mm_dist": float(np.linalg.norm(me - te, axis=1).mean())}
print("matcher loaded (epoch", mck["epoch"], ")")

In [ ]:
# 6. Today's text model (original 8x-data model, guidance s = 4) and notebook 12's bone fix
e = tokenizer([""], padding="max_length", truncation=True, max_length=B.MAX_TOKENS, return_tensors="np")
EMPTY_IDS, EMPTY_ATTN = e["input_ids"][0].astype(np.int64), e["attention_mask"][0].astype(np.int64)
ck = torch.load(os.path.join(ROOT, "checkpoints", f"t2c_more_{B.N_POINTS}pts", "best_loss.pt"), map_location=DEVICE, weights_only=False)
CFG = ck["config"]
model = T.Text2Codes(n_codes=CFG["n_codes"], d_model=CFG["d_model"], n_heads=CFG["n_heads"], n_enc=CFG["n_enc"],
                     n_dec=CFG["n_dec"], ff=CFG["ff"], dropout=CFG["dropout"], max_codes=CFG["max_codes"],
                     text_encoder=CFG["text_encoder"]).to(DEVICE)
model.load_state_dict(ck["model"]); model.eval()
S_GUIDE = 4.0
base_test = [c[:, 0] for c in codes["test"]]
data_test = T.CodeTextData(texts["test"], base_test, tokenizer, CFG["max_codes"])
tidx = list(range(len(test["poses"])))
perm = np.random.RandomState(0).permutation(len(tidx))
shuf = T.CodeTextData([texts["test"][j] for j in perm], base_test, tokenizer, CFG["max_codes"])

@torch.no_grad()
def generate_cfg(d, idx, s=1.0, temperature=1.0, min_codes=2, bs=64):
    """Sample codes with classifier-free guidance: logits = general + s * (sentence - general)."""
    model.eval(); out = []
    for b0 in range(0, len(idx), bs):
        ch = np.asarray(idx[b0:b0 + bs]); n = len(ch)
        ids = torch.as_tensor(d.ids[ch], device=DEVICE); attn = torch.as_tensor(d.attn[ch], device=DEVICE)
        if s != 1.0:
            ids = torch.cat([ids, torch.as_tensor(np.repeat(EMPTY_IDS[None], n, 0), device=DEVICE)])
            attn = torch.cat([attn, torch.as_tensor(np.repeat(EMPTY_ATTN[None], n, 0), device=DEVICE)])
        mem, mem_pad = model.encode_text(ids, attn)
        seq = torch.full((len(ids), 1), model.BOS, dtype=torch.long, device=DEVICE)
        done = torch.zeros(n, dtype=torch.bool, device=DEVICE)
        lengths = torch.full((n,), model.max_codes, dtype=torch.long, device=DEVICE)
        for t in range(model.max_codes):
            lg = model.decode_step(seq, mem, mem_pad)[:, -1].float()
            if s != 1.0:
                lg = lg[n:] + s * (lg[:n] - lg[n:])
            if t < min_codes:
                lg[:, model.END] = float("-inf")
            nxt = torch.multinomial(F.softmax(lg / temperature, -1), 1)[:, 0]
            ended = (nxt == model.END) & ~done
            lengths[ended] = t; done |= ended
            nxt = nxt.masked_fill(done, model.PAD)
            seq = torch.cat([seq, (torch.cat([nxt, nxt]) if s != 1.0 else nxt)[:, None]], 1)
            if bool(done.all()):
                break
        sq = seq[:n, 1:].cpu().numpy()
        out += [sq[k, :int(lengths[k])].astype(np.int64) for k in range(n)]
    return out

lim = json.load(open(os.path.join(RESULTS_DIR, "12_bone_limits.json")))
EDGES = [(b["from"], b["to"], b["name"]) for b in lim["bones"]]; LIMIT = np.array([b["max"] for b in lim["bones"]], np.float32)
SH_LIMIT = lim["shoulder_width_max"]; bd = L.layout(POINT_IDX)["body"]; SH = (bd["l_shoulder"], bd["r_shoulder"])
ARM_EDGES = sum(1 for _, _, n in EDGES if "finger" not in n)

def fix_bones(p):
    # notebook 12: shorten any bone longer than its real limit, keeping its direction; points further out move with it
    p = np.asarray(p, np.float32)[..., :2]
    q = p.copy(); valid = np.abs(p).sum(-1) > 0
    shift = np.zeros_like(p)
    a, b = SH
    ok = valid[:, a] & valid[:, b]
    v = p[:, b] - p[:, a]; d = np.linalg.norm(v, axis=1)
    sc = np.where(ok & (d > SH_LIMIT), SH_LIMIT / np.maximum(d, 1e-6), 1.0)[:, None]
    mid = (p[:, a] + p[:, b]) / 2
    q[:, a] = np.where(ok[:, None], mid - v / 2 * sc, p[:, a]); q[:, b] = np.where(ok[:, None], mid + v / 2 * sc, p[:, b])
    shift[:, a], shift[:, b] = q[:, a] - p[:, a], q[:, b] - p[:, b]
    for (a, b, _), lim_ in zip(EDGES, LIMIT):
        v = p[:, b] - p[:, a]; d = np.linalg.norm(v, axis=1)
        sc = np.minimum(1.0, lim_ / np.maximum(d, 1e-6))[:, None]
        new = np.where(valid[:, [a]], q[:, a] + v * sc, p[:, b] + shift[:, a])
        q[:, b] = np.where(valid[:, [b]], new, p[:, b])
        shift[:, b] = q[:, b] - p[:, b]
    return q

@torch.no_grad()
def fill_extra(d, idx, base, temperature=0.0, bs=128):
    out = []
    for s in range(0, len(idx), bs):
        ch = np.asarray(idx[s:s + bs])
        out += pred.fill(torch.as_tensor(d.ids[ch], device=DEVICE), torch.as_tensor(d.attn[ch], device=DEVICE),
                         [base[k] for k in range(s, s + len(ch))], temperature)
    return out
print("text model epoch", ck["epoch"], "| guidance s =", S_GUIDE, "| bone limits:", len(EDGES), "bones")

In [ ]:
# 7. The default pipeline from notebook 16: length predictor + window 0.8-1.25
def len_feats(tx, bs=1024):
    e = st.encode(list(tx), batch_size=bs, convert_to_numpy=True, show_progress_bar=False).astype(np.float32)
    w = np.array([[len(t.split()), len(t)] for t in tx], np.float32)
    return np.concatenate([e, np.log1p(w)], 1)

lck = torch.load(os.path.join(ROOT, "checkpoints", "length_predictor_v1.pt"), map_location=DEVICE, weights_only=False)
mu_x, sd_x = lck["mu"], lck["sd"]
lenp = nn.Sequential(nn.Linear(len(mu_x), 256), nn.GELU(), nn.Dropout(0.2), nn.Linear(256, 1)).to(DEVICE)
lenp.load_state_dict(lck["model"]); lenp.eval()

@torch.no_grad()
def predict_frames(tx):
    x = torch.as_tensor((len_feats(tx) - mu_x) / sd_x, device=DEVICE)
    return np.clip(np.exp(lenp(x)[:, 0].cpu().numpy()), 8, 300)

@torch.no_grad()
def generate_len(d, idx, s, lo, hi, temperature=1.0, bs=64):
    """Like generate_cfg, but clip k may not end before lo[k] codes and must end by hi[k] codes."""
    model.eval(); out = []
    lo, hi = np.asarray(lo), np.asarray(hi)
    for b0 in range(0, len(idx), bs):
        ch = np.asarray(idx[b0:b0 + bs]); n = len(ch)
        lo_t = torch.as_tensor(lo[b0:b0 + n], device=DEVICE); hi_t = torch.as_tensor(np.minimum(hi[b0:b0 + n], model.max_codes - 1), device=DEVICE)
        ids = torch.as_tensor(d.ids[ch], device=DEVICE); attn = torch.as_tensor(d.attn[ch], device=DEVICE)
        ids = torch.cat([ids, torch.as_tensor(np.repeat(EMPTY_IDS[None], n, 0), device=DEVICE)])
        attn = torch.cat([attn, torch.as_tensor(np.repeat(EMPTY_ATTN[None], n, 0), device=DEVICE)])
        mem, mem_pad = model.encode_text(ids, attn)
        seq = torch.full((len(ids), 1), model.BOS, dtype=torch.long, device=DEVICE)
        done = torch.zeros(n, dtype=torch.bool, device=DEVICE)
        lengths = torch.full((n,), model.max_codes, dtype=torch.long, device=DEVICE)
        for t in range(model.max_codes):
            lg = model.decode_step(seq, mem, mem_pad)[:, -1].float()
            lg = lg[n:] + s * (lg[:n] - lg[n:])
            lg[t < lo_t, model.END] = float("-inf")                      # too early to stop
            force = t >= hi_t                                            # long enough: stop now
            lg[force] = float("-inf"); lg[force, model.END] = 0.0
            nxt = torch.multinomial(F.softmax(lg / temperature, -1), 1)[:, 0]
            ended = (nxt == model.END) & ~done
            lengths[ended] = t; done |= ended
            nxt = nxt.masked_fill(done, model.PAD)
            seq = torch.cat([seq, torch.cat([nxt, nxt])[:, None]], 1)
            if bool(done.all()):
                break
        sq = seq[:n, 1:].cpu().numpy()
        out += [sq[k, :max(1, int(lengths[k]))].astype(np.int64) for k in range(n)]
    return out

def finish(d, idx, base):
    """base codes -> + 3 extra codes (random, T = 1) -> poses -> bone fix (the default pipeline)."""
    return [fix_bones(p) for p in codes_to_poses_rvq(rvq, fill_extra(d, idx, base, temperature=1.0), DEVICE)]


def default_pipeline(d, idx, texts_, seed=0):
    torch.manual_seed(seed)
    Pc = np.maximum(2, np.round(predict_frames(texts_) / 4)).astype(int)
    base = generate_len(d, idx, S_GUIDE, np.floor(Pc * 0.8).astype(int), np.ceil(Pc * 1.25).astype(int))
    return finish(d, idx, base)
print("default pipeline ready")

In [ ]:
# 8. Notebook 18's depth model (depth_lifter_v1, best pass) and lift(): x, y -> depth of all 123 points
MAX_FRAMES = 300
class DepthNet(nn.Module):
    def __init__(self, n=B.N_POINTS, d=256, layers=4, heads=4, maxT=MAX_FRAMES, dropout=0.1):
        super().__init__()
        for k in ("xm", "xs", "vs"):                                 # input scaling; the real values come from the checkpoint
            self.register_buffer(k, torch.ones(n, 2))
        self.inp = nn.Linear(n * 5, d); self.pos = nn.Embedding(maxT, d)
        self.conv = nn.Conv1d(d, d, 5, padding=2)
        self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, batch_first=True, norm_first=True), layers,
                                         enable_nested_tensor=False)
        self.out = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, n))
    def forward(self, xy, pres, pad):
        pm = pres.unsqueeze(-1).float()
        v = torch.zeros_like(xy); ok = (pres[:, 1:] & pres[:, :-1]).unsqueeze(-1)
        v[:, 1:] = (xy[:, 1:] - xy[:, :-1]) * ok / self.vs
        x = torch.cat([(((xy - self.xm) / self.xs) * pm).flatten(2), v.flatten(2).clamp(-10, 10), pres.float()], -1)
        h = self.inp(x) + self.pos(torch.arange(x.shape[1], device=x.device))
        h = h + self.conv(h.transpose(1, 2)).transpose(1, 2)
        return self.out(self.enc(h, src_key_padding_mask=pad))

dck = torch.load(os.path.join(ROOT, "checkpoints", "depth_lifter_v1", "best.pt"), map_location=DEVICE, weights_only=False)
dnet = DepthNet().to(DEVICE); dnet.load_state_dict(dck["model"]); dnet.eval()
ZMEAN_T, ZSTD_T = (torch.as_tensor(np.asarray(a, np.float32), device=DEVICE) for a in (dck["zmean"], dck["zstd"]))
print("depth model: pass", dck["epoch"] + 1)

@torch.no_grad()
def lift(xys, press, bs=32, win=MAX_FRAMES, step=200):
    """lists of (T, 123, 2) x, y and (T, 123) tracked flags -> list of (T, 123) depth in shoulder widths.
    Long sequences (a whole paragraph) are read in overlapping 300-frame windows and blended (as in notebook 18)."""
    pieces = []
    for i, x in enumerate(xys):
        T_ = len(x)
        starts = [0] if T_ <= win else list(range(0, T_ - win, step)) + [T_ - win]
        pieces += [(i, a, min(T_, a + win)) for a in starts]
    acc = [np.zeros((len(x), B.N_POINTS), np.float64) for x in xys]
    wsum = [np.zeros((len(x), 1), np.float64) for x in xys]
    for b0 in range(0, len(pieces), bs):
        ch = pieces[b0:b0 + bs]; Tm = max(e - a for _, a, e in ch)
        xy = np.zeros((len(ch), Tm, B.N_POINTS, 2), np.float32); pr = np.zeros((len(ch), Tm, B.N_POINTS), bool); pad = np.ones((len(ch), Tm), bool)
        for k, (i, a, e) in enumerate(ch):
            xy[k, :e - a] = np.asarray(xys[i], np.float32)[a:e, :, :2]; pr[k, :e - a] = np.asarray(press[i])[a:e]; pad[k, :e - a] = False
        xy[~pr] = 0
        t = lambda v: torch.as_tensor(v, device=DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            pred = dnet(t(xy), t(pr), t(pad)).float()
        z = (pred * ZSTD_T + ZMEAN_T).cpu().numpy()
        for k, (i, a, e) in enumerate(ch):
            n_ = e - a; tt = np.arange(n_)
            w = np.minimum(np.minimum(tt + 1, n_ - tt), 50).astype(np.float64)[:, None]
            acc[i][a:e] += z[k, :n_] * w; wsum[i][a:e] += w
    return [(a / np.maximum(w, 1e-9)).astype(np.float32) for a, w in zip(acc, wsum)]

## Translation: IndicTrans2 stays loaded

Notebook 19 started IndicTrans2 for each batch. For a live demo it must answer in about a second, so here it runs as one small
background process that loads the Hindi/Kannada → English model **once** and then waits for sentences. Translations already made in
notebook 19 (the demo paragraphs) come from `translations19/cache.json` instantly, and new ones are added to it.

In [ ]:
# 9. IndicTrans2 as a background worker (model loaded once)
import re, hashlib, threading
IT2_LIB = "/content/it2lib" if not os.environ.get("BAHU_ROOT") else os.path.join(ROOT, "_it2lib")
IT2_WORKER = os.path.join(os.path.dirname(IT2_LIB), "it2_worker.py")
INDIC_EN = "ai4bharat/indictrans2-indic-en-1B"
LANGS = {"en": "eng_Latn", "hi": "hin_Deva", "kn": "kan_Knda"}
TR_DIR = os.path.join(ROOT, "translations19"); os.makedirs(TR_DIR, exist_ok=True)
TR_CACHE_F = os.path.join(TR_DIR, "cache.json")
TR_CACHE = json.load(open(TR_CACHE_F)) if os.path.exists(TR_CACHE_F) else {}
print("translations already saved:", len(TR_CACHE))

if not os.path.exists(os.path.join(IT2_LIB, "transformers", "__init__.py")):
    t0 = time.time()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--target", IT2_LIB, "transformers==4.46.3",
                    "tokenizers==0.20.3", "huggingface_hub==0.26.2", "IndicTransToolkit==1.1.1"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacremoses", "indic-nlp-library-itt", "sentencepiece"], check=True)
    print(f"IndicTrans2 libraries installed [{time.time() - t0:.0f}s]")

open(IT2_WORKER, "w").write(r"""
import sys, json, time, types
import torch
try:
    from IndicTransToolkit.processor import IndicProcessor
except ImportError:                              # newer transformers break the toolkit's training helpers; we only need the processor
    for n in ("collator", "evaluator"):
        m = types.ModuleType("IndicTransToolkit." + n); m.IndicDataCollator = m.IndicEvaluator = None
        sys.modules["IndicTransToolkit." + n] = m
    from IndicTransToolkit.processor import IndicProcessor
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
name = sys.argv[1]
dev = "cuda" if torch.cuda.is_available() else "cpu"
ip = IndicProcessor(inference=True)
def say(obj):
    sys.stdout.write(json.dumps(obj, ensure_ascii=False) + "\n"); sys.stdout.flush()
try:
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(name, trust_remote_code=True,
                                                torch_dtype=torch.float16 if dev == "cuda" else torch.float32).to(dev).eval()
except Exception as e:
    say({"ready": False, "error": f"{type(e).__name__}: {e}"}); sys.exit(1)
say({"ready": True, "seconds": round(time.time() - t0, 1)})
for line in sys.stdin:
    try:
        job = json.loads(line)
        out = []
        for b0 in range(0, len(job["sents"]), 8):
            batch = ip.preprocess_batch(job["sents"][b0:b0 + 8], src_lang=job["src"], tgt_lang=job["tgt"])
            enc = tok(batch, truncation=True, padding="longest", return_tensors="pt", return_attention_mask=True).to(dev)
            with torch.no_grad():
                gen = mdl.generate(**enc, use_cache=True, min_length=0, max_length=256, num_beams=5, num_return_sequences=1)
            dec = tok.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=True)
            out += [d.strip() for d in ip.postprocess_batch(dec, lang=job["tgt"])]
        say({"id": job["id"], "out": out})
    except Exception as e:
        torch.cuda.empty_cache()
        say({"id": job.get("id") if isinstance(job, dict) else None, "error": f"{type(e).__name__}: {e}"})
""")

class IT2Worker:
    """One background IndicTrans2 process; translate() sends sentences and waits for the answer (one request at a time)."""
    def __init__(self):
        self.lock = threading.Lock(); self.n = 0; self.p = None
    def start(self):
        env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
        env["PYTHONPATH"] = IT2_LIB + os.pathsep + env.get("PYTHONPATH", "")
        self.errf = open(os.path.join(os.path.dirname(IT2_LIB), "it2_worker_err.txt"), "w")
        self.p = subprocess.Popen([sys.executable, IT2_WORKER, INDIC_EN], env=env, stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                  stderr=self.errf, text=True, bufsize=1)
        first = json.loads(self.p.stdout.readline() or '{"ready": false, "error": "the worker stopped"}')
        if not first.get("ready"):
            err = first.get("error", "") + "\n" + open(self.errf.name).read()[-3000:]
            if any(k in err.lower() for k in ("gated", "restricted", "403", "401")):
                raise RuntimeError(f"Hugging Face refused the IndicTrans2 download: open https://huggingface.co/{INDIC_EN}, click agree, "
                                   "check HF_TOKEN, then run this cell again.\n" + err[-800:])
            raise RuntimeError("IndicTrans2 could not start; paste this to Claude:\n" + err[-3000:])
        print(f"IndicTrans2 loaded in {first['seconds']}s and waiting for sentences")
    def translate(self, sents, src, tgt="eng_Latn"):
        with self.lock:
            if self.p is None or self.p.poll() is not None:
                print("(re)starting IndicTrans2 ..."); self.start()
            self.n += 1
            self.p.stdin.write(json.dumps({"id": self.n, "src": src, "tgt": tgt, "sents": sents}, ensure_ascii=False) + "\n"); self.p.stdin.flush()
            ans = json.loads(self.p.stdout.readline() or '{"error": "IndicTrans2 stopped (out of memory?)"}')
        if "error" in ans:
            raise RuntimeError("translation failed: " + ans["error"])
        return ans["out"]

def _key(model, src, tgt, s):
    return hashlib.md5(f"{model}|{src}|{tgt}|{s}".encode("utf-8")).hexdigest()
_cache_lock = threading.Lock()
def translate(sents, lang):
    """Hindi/Kannada sentences -> English (cached on Drive)."""
    s_ = LANGS[lang]
    todo = [x for x in dict.fromkeys(sents) if _key(INDIC_EN, s_, "eng_Latn", x) not in TR_CACHE]
    if todo:
        for x, y in zip(todo, IT2.translate(todo, s_)):
            TR_CACHE[_key(INDIC_EN, s_, "eng_Latn", x)] = y
        with _cache_lock:
            json.dump(TR_CACHE, open(TR_CACHE_F + ".tmp", "w"), ensure_ascii=False); os.replace(TR_CACHE_F + ".tmp", TR_CACHE_F)
    return [TR_CACHE[_key(INDIC_EN, s_, "eng_Latn", x)] for x in sents]

IT2 = IT2Worker(); IT2.start()
t0 = time.time(); print("test:", IT2.translate(["आज मौसम अच्छा है।"], "hin_Deva"), f"[{time.time() - t0:.1f}s]")

In [ ]:
# 10. One paragraph -> one avatar clip (the notebook-19 steps: split, translate, cut long sentences, generate, join, depth)
JOIN = "ease 8"                                   # chosen in notebook 19 (smallest jump that keeps the meaning scores)
MAX_PIECES = 12                                   # a safety limit for the live demo (about 12 sentences)
def split_sentences(text):
    text = re.sub(r"\s+", " ", text.strip())
    parts = re.split(r"(?<=[.!?])\s+|(?<=[।॥])\s*", text)
    return [p.strip() for p in parts if p and re.search(r"\w", p)]
nw = np.array([len(t.split()) for t in texts["train"]])
MAX_WORDS = int(np.clip(np.percentile(nw, 90), 10, 22))
JOINERS = {"and", "but", "because", "so", "while", "which", "who", "where", "when", "after", "before", "or", "then", "although"}
def chunk_english(s, max_words=None):
    max_words = max_words or MAX_WORDS
    w = s.split()
    if len(w) <= max_words:
        return [s]
    cands = [i for i in range(3, len(w) - 2) if w[i - 1][-1] in ",;:" or w[i].lower() in JOINERS]
    cut = min(cands, key=lambda i: abs(i - len(w) / 2)) if cands else len(w) // 2
    return chunk_english(" ".join(w[:cut]).rstrip(",;:"), max_words) + chunk_english(" ".join(w[cut:]), max_words)

bd = LAY["body"]; ARM_HAND = [bd[k] for k in ["l_shoulder", "r_shoulder", "l_elbow", "r_elbow", "l_wrist", "r_wrist"]] + \
     list(range(LAY["parts"]["hands"].start, LAY["parts"]["hands"].stop))
def smooth01(n):
    t = (np.arange(n) + 1) / (n + 1)
    return t * t * (3 - 2 * t)
def join(clips, method):
    clips = [np.asarray(c, np.float32)[..., :2] for c in clips]
    out, bounds = clips[0].copy(), [[0, len(clips[0])]]
    for c in clips[1:]:
        if method.startswith("ease"):
            m = min(int(method.split()[1]), max(1, len(c) // 2))
            c = c.copy(); c[:m] += (out[-1] - c[0])[None] * (1 - smooth01(m))[:, None, None]
        bounds.append([len(out), len(out) + len(c)]); out = np.concatenate([out, c])
    return out, bounds
def join_jumps(seq, bounds):
    x = np.asarray(seq, np.float32)[:, ARM_HAND]
    sp = np.linalg.norm(x[1:] - x[:-1], axis=-1).mean(1)
    typ = max(float(np.median(sp)), 1e-4)
    return [round(float(sp[max(0, b0 - 3):b0 + 2].max()) / typ, 2) for b0, _ in bounds[1:]]
def r3(a):
    return [[round(float(v), 3) for v in row] for row in np.asarray(a, np.float32).reshape(len(a), -1)]

def detect_lang(text):
    kn = len(re.findall(r"[ಀ-೿]", text)); hi = len(re.findall(r"[ऀ-ॿ]", text))
    return "kn" if kn > max(hi, 2) else "hi" if hi > 2 else "en"

GPU_LOCK = threading.Lock(); SEEN = {}; N_LIVE = [0]
def make_paragraph(text, lang="auto"):
    text = re.sub(r"\s+", " ", (text or "").strip())[:1500]
    if not text:
        raise ValueError("empty text")
    lang = detect_lang(text) if lang == "auto" else lang
    if lang not in LANGS:
        raise ValueError("language must be en, hi or kn")
    if (lang, text) in SEEN:
        return {**SEEN[(lang, text)], "timing": {"cached": True}}
    t0 = time.time()
    sents = split_sentences(text) or [text]
    en = sents if lang == "en" else translate(sents, lang)
    segs = [{"src": s, "en": piece} for s, e in zip(sents, en) for piece in chunk_english(e)][:MAX_PIECES]
    t1 = time.time()
    seg_en = [g["en"] for g in segs]
    with GPU_LOCK:
        d_ = T.CodeTextData(seg_en, [np.zeros(2, np.int64)] * len(seg_en), tokenizer, CFG["max_codes"])
        poses = default_pipeline(d_, list(range(len(seg_en))), seg_en, seed=19)
        t2 = time.time()
        variants = {}
        for mth in (JOIN, "cut"):
            seq, bnd = join(poses, mth)
            z = lift([seq], [np.ones(seq.shape[:2], bool)])[0]
            variants[mth] = {"n": len(seq), "pts": r3(seq), "z": r3(z), "jumps": join_jumps(seq, bnd),
                             "segments": [{**g, "start": int(a), "end": int(b)} for g, (a, b) in zip(segs, bnd)]}
    t3 = time.time()
    N_LIVE[0] += 1
    clip = {"uid": f"live{N_LIVE[0]}_{lang}", "kind": "paragraph", "lang": lang, "source": text,
            "text": " ".join(seg_en), "join": JOIN, **variants[JOIN], "cut": variants["cut"]}
    res = {"ok": True, "clip": clip, "point_idx": [int(v) for v in POINT_IDX], "fps": 25,
           "timing": {"translate": round(t1 - t0, 1), "generate": round(t2 - t1, 1), "depth": round(t3 - t2, 1)}}
    SEEN[(lang, text)] = res
    print(f"[{time.strftime('%H:%M:%S')}] {lang}: {text[:60]!r} -> {len(segs)} pieces, {clip['n'] / 25:.1f}s of signing, "
          f"took {t3 - t0:.1f}s (translate {t1 - t0:.1f}, generate {t2 - t1:.1f}, depth {t3 - t2:.1f})", flush=True)
    return res

# warm-up: one English and one Hindi sentence (the first run of each step is the slowest)
for txt in ["Thank you for coming today.", "आप सभी का स्वागत है।"]:
    r_ = make_paragraph(txt)
print("live pipeline ready")

## Start the server and open the avatar page

The link opens the BahuSanket front end. It works in this browser while this Colab tab is open. If it does not open, use the optional
cell after it.

In [ ]:
# 11. Web server on port 8000: the avatar page + /api/generate
import gzip, base64, urllib.request, http.server, socketserver
LIVE_DIR = "/content/live20" if not os.environ.get("BAHU_ROOT") else os.path.join(ROOT, "_live20")
os.makedirs(LIVE_DIR, exist_ok=True)
open(os.path.join(LIVE_DIR, "viewer.html"), "wb").write(gzip.decompress(base64.b64decode("...")))
open(os.path.join(LIVE_DIR, "app.html"), "wb").write(gzip.decompress(base64.b64decode("H4sIAPKL/moC/7V9W28cSXbmu35FqDweZ7WyklXFi0iWSA2lplqyqW4uqemeMUGos6qiqrKZlVmdmUWypCZgG1jPPqz3wQvvArt+WUBjeBeG15CxWPvFBrx/RPAv2fOduGRkXSiqZ9wNsTLjcuLEiRPnFpd8dL+f9orZRIpRMY737z3Cj4jDZLhXk0mNEsayCEVvFGa5LPZq02LQ2LbJSTiWe7XLSF5N0qyoiV6aFDKhYldRvxjt9eVl1JMNfvFFlERFFMaNvBfGcq8FIEVUxHL/STianobJhSzEUXQpxWk0TGT2aE3l3nsUR8mFyGS8V5tkkppIZI/aGmVysFcbFcUk311bG1DLeTBM02Esw0mUB710XKvUzYtZLPORlHequ9bL8/bjQTiO4tneQXERJXmaPHhOhMpiOYy6sdy9Go6Kn200m52HzeZPdcknWdRL43AoH3yRpYXMv5/K3XSSv/G5cKsdBDtb/ibV0Y9O1RdPXj44juX1g5dpkpbAN8sSX6ZF+oAIlT/4XF6GSTgMs6gsuLWs4B+ESRL2w2qpfpRP4nC2l1+FE9CIKbN/T4jdLE0L8ZaehGg0usNd8TuD9mBj0OropHyaDcKeRDr/Z9KJyJTW6rbb7aZJG08L2afUzXBra6ttUmk8UL2/2e/1ty3YgijWKNIJ5cgN2ZUPqzndtCjSMWX2dvobPWkyw16PmI2S292t7uZcssapiqdGvtvceLhuE+NRmADR9mBr0LPoZzq1t/MwbG2Y1G7an1HixmAz3LRIgn8amqi7omZ5QFgeqPmidiqHqRQ/f0HP+Swv5LgxjeiRRqmRyywaVKCpdmqG8YTLeJ8MbAyGEjViMAEGE2AwqjiNOCefEFEA8hnSGydyOI3DjPJfyiROfWHLAOYN/fvZWPajUHg0GQcyyxvU2TSjWT2SYyJuP8wu6pqHmJ92k7TwzvphETYKFNmrxRFxY+3clLK81mq1tojUFT5r7bSbrX7H8pjckj2503H4aycMt8N2p+Stdm99a10z3Dx3tTfWm+1uZ5GzWhutbmu743LVw3ZvI5SdeY5q9ltha6sErxGVm9vhdrPj8NPDQbc54BTDS4Nmd32z1yn5KNzpbnabBtYiIVXOjSY8k7NCSZSpnVdn7L8BFf9Nafjbo+By+oFyn4m3opteN/LoTZQQkbpp1pcZwbjucD6Uni8AEgXD3sUwS6do8DLMPNC13lHQTQp1g5LGYTaMkl3RVFBUfUaF552SxqaKndd1mmyQ0AISWpSivFZJ15K7VpnVCi71gvrX2poQ9hitxkhiRlFSsKkoMQn7fe5oa3tyTWheeyjtC5lcenk4kI0wkyF1glR6I0NdXzQn1/W6aH+8fCwHprghb3CVhROiHFVUyp4abm83gZ8lkQinRdoRVkwOs4gYchhOFJKKggFxGMGxhQYkrjoiJIGRNCKScvmuABfJzNTcQE2UagCFXYG/GlQ3I5b5CLBumEtQ0IBrfwzcqEUQVw2ubqmuh+lKjwrp+MrA9eJwPPHaTOD1oH15RT/0UuWnWBbUzQYELw9kI2i25LiCi6K4rVHhT57WdV18EsWxLcvjt6tHY1VPIPPrVW5j0li+2iQ2UUl6KmVhP5oSRXd2dspUqkbl8jSO+ho6iF1fgasdqChhrv7Y4Jdcgw7u7nblIM0kdVRboaTyah2h+ZELm3myvQTxzebvdpZM/QVCBqSN31Y7oERb3YJcnulCcJBdbNIBRxJrFPbTK8wf/L9O5FRSbhxde1Ei8mzY9av1RHvzd31REIsQ82RO2+MwStz5oGcg/W0QdSmtkEB9Ok6IHOMogRho+qI1yOrua9DapJTK1K2MESmFrOhULAVHLuw0ITkIDYPOndoncKoPvTDrLyWa1nX1j7Le3LArKVqRlyuEFIsaQsv0RAt9hdKo/ZvLBdX2nE4Jcjl0B60yN1Z1dZ4h5yan7S1PYO7dLYIPCHSnpOIT3UUgMSJ1VMz1Zwv9MSg1q5PJYcfOisljsdqCbNlYLVt60yxH/UkaKWkwh+YZ6dKwQdZpnsv+Xq3IyP4+v32mLcFIq3jALuR1AQVIMIx2a0JaLCOGHsqdpbrZ4bP2qk66UnUVE7OOv7uZQpRgrC5lVkTkfCs2tohtNq0gHaaLClNxf3OyShYv5Ro1EsEki4ifZx9hnIWJ8LA6J0GE9tanKxszvCvIdYfRX8pqum/DUZoXK3qm+7H+29SZd5tOH8V7l4Y3JC8SYjSFdVHQSAebm06lLB1i8lRqDdLeNG9cRjk8ULJJZSx7xXyqmSjVdDQ0LZRbsW779lE1trGoxjSYRjoYkCWqpRYz7kW3f4sAXmXOrDaZ5DXZaLHMF6fDArvz/NhaOT2q4KgvRoxWmGQpJtpgy9QsKTtL/v4EmMVRTlAQvNkVSQor1lEelu2anTuq/ExOZFh4G/6c8q27fVyh1jfbRq0b5G5vpL2kkRu3d3HELtsnqfJmZbrBk1FJC9SeE8zrt3GCRafr8kI3TnsXS6feQmNVKEHxI/i0BBF0p/lMESdMSLYWEc1NMZnGuaSObORCkkdDmDRoqpBMGiDmKpcB2E2KUaM3iuK+16670Bp9yT0M2hAAq2utL6+1cXutjeW1tvIKjn1iZpcDbrGof3YhZ4MsHNPcUlR4C0v+ozWXBrIy2Z/2ZJ+GQZFVvZdcvZz4aubdoblgnA+rLps70BurxYCrrIMN6wUSuEBmWemN/E53vb3VMsM9ilg9LbNlt5SheW3Btrd4oqRkIQxiuBvKPSzhWAMQIr7Bgm5XwJO81QraWJyQy/SfmqWfoP5uN3cWlN8yEpTzyumfshypfsaLGKXleNvgul4aR0hYycFbW+YAcmv5OGSPfOlwf4JwaDnCIRxKArdiwBWFlzgvXI8qTdI8MkxPAju6lCs8pZJFRlG/LxPSevmETAEqRvV3xYZYExvBRqdqKbu8RgZoz6Pky5FokKnXrEY+FOctWGxggjBrDIEMUdNrbTf7cmhMBhudrFdTVGSy/nGnkKVBPp1gFSsXXrVLLS0BFKV0v4iW6ASZKNyfuQ6JzwwhKooNEHphchnmS3RJhWB2vvObqV5M88pYhV3qyxQCnhlvma6n556ct0ncSWzSSqdwtST6BNZ0VR/wPlPscu4L8+SSADJU3I/GoH+YFNaznqCfK7qs4lhKoGgTyUgXFY+eC1lhDtzici2w3EoD1YQaxE5zwUL9CJ/dLolMf4PRnInY/rEjUgWb3yFG8ZvGpivBgIWgJ0IcvmgH6wh6gtHqy4yxKtZyjhibt1vuut4uWZ7FbJ7HTPw0zO5q2N/i+H43zYtoMGvYcGPF5Ecbyku63Ues6sgta7T+xj7ifFxsiaZUIfdoLD/NJl2/nR0voUXpN5mOqbs98lbDLpYUkZBXtNBWb6SRGPDC8x2Zfj5ExpXDlVHZm3uP1vQS9yOsv9BPP7oER+b5Xg1jXcPit5tIqoTTqqkcdtfplDNqOTsXHq3Rq8mZ7L8iCStourxIyMJMeEeDOAqT4RQq4F/+rzhMhuS9jfD44d0/fnj3zx/e/fWHX//Zh3d/9eHXf8Sp7//iw/u//vD3f4a/7//Ho7WJxmeNENKP7Exq3BBYromor5/2n6ptEsRU//pHf0X9p6LcSVV9rrsIxS7pLyKbTnfb+6+wTyQkbs1C0sSTEXW6bfOdirkc1kSWxnKvBm6c1AQbVnHYxUYMQweFLraanFJ5A4cgaWMTu1KI6PxSE7zsqfalwETQIKtRvv3PZUGdfrSmKn0CSJnMAxyE5E3U9vVA/QiQo2gFyMUB/xHQL1YhvMA4VdAO/9CLjWxiJPBSUwbDKI1JzOzVeMDTjEacHCABJU0vYyxc5JB1SY/cLtKOmki+eB4Rv6OM1hDEe9WxPzasg8mR06yo7T9aM1gsY6VhuoQzDM+r6KLiIypYpdY+T7poGTe4E+eiSzxOGD8tslg8oK4U2Hpk5ssiyRzcTFSH+sDlX2Wz3WrVVQNpOocQoh5Wea02XH0iw62ARMz3G/DZCqDEc5/GXuSPGIkA/1kNlH50ueKbUViIUTiZyCR3hzuO9h9191uQh69gZiGURC3uQ6r2OKUt/vVP/9xK08rAGszZkdOVwW08Po/WCPZ8Q2009IVMZKbbAWNyA+BUGKHE7qvaGOp6tzexjiaexDLpA345ib4jX1X2VwLvosbtkDcA+XM5KcA6++3PGe/1z1eC7KPoUpCP1tK4VGVWR+RDNX78oIS7Mu3NWJLTSHOSjBJFhImjOQTrX5JRVr/DjFNTlLqdp2IQZqu0CVxm1TQ/7bsq0KqzOQWk3eHaUu1UWOWjHh1aau+MxeEok7LKp0TO8JLolzFDkG4FMqrKchhkIPYuatpVroA6RY4YRMMpCdN0IIoRydSQLDFwmQpd5EuhVzvC5Nc9wfP+URrCkmR4ClfW/86snNfxbC4rIOZlvzqNV8k/MnFXC2ctPNgiISt7Xjgfh9NcrpZHi0AyuQzMCad+RMTDwlVA+Gm/GTRFviCotbHOxJxI2a8O/akacKGyymqQcspRuQzjKWHWIq5GGdH6f/+VZhLn3VK8GTzcrGlHQfZ1VSTesfZmzdZZWoW6ybBXcNCthtrpKL2qdnW1VYI9wGSZMcctt80OOG9xqO4AVk+jpfbOcTlflrDBaj62gg3eQ23fiP2+6M548jibksdpX8Y+OZCFwP5j8sD6RgAEQnVrVzwK53YVD6NiNO3yXuLLbNzo8V8ElqJB1OPY8VpBEmZtDNsqW8uVLbH29cnL1muy3nNSWtSp16+uSOi9PuVc4n5yfbAN+3WXzEAiCe9sTtJ0AvRr+6gsysqCKwtV+dFauC/+5X+KSXRNDPAi6QXCo/LieNqNo544ishd4lWEZj2w6pZjLgtdCzMCEaTZcC3s5mvt9eZm0Npabz78KH6O4mZ0vIMXG09GZBMWqs2I5xhGfkmro+mQ3L4h3Fmi6hpK5bLI1w6vJ+T2qQWFo7C7FmGufhQVLgUkgkXfSj+Yn7yXRZNCs6cKUo3ZaXwr9Gte2+WNf1pp7AqLdK+fBN/lxD/RZRYkslhLJuM1LvWzZtDaJlFETBvFfZUWEKtNY0k1ar4Ftxb2+zSka58I1lina9/l4zUF7mc89qpcg9jxoxDnK/xsPdgMNslW6JZpBmfy64H3vRvlcDPN5omnihLlFNnEZ4IU5avnJ4eHYpClY0O/jsl/K744evUMOk1m4sYtY4gSc16+VpYDEg4AYnGVfhxPiX18JPy8iOLcwlugSufevR7mkPiJ2BNe1K+LvX3RT3vTMdlrATHVYSzx+GT2oo/sji6udPBhTLV+4hmNTLn31tZEw/4n1Eh/lwsVEC5zNBhtPzAQRQ/bQkZmoMyIFnsikVeKcsE3svvF0YnO8d7q+r4IExy3iMIc+wWmkhLiySg0L5CjMruUn2fhFc2qJ9PBAIElZIobatA0FdAcO46uZXyCGea9DIsRhtqjSv30KlBHPMoCvmjX3drptJhMi6cIzJxiEzlhrrA+PfniSZlsSUhCSFZ6d4oUz1KgRzZSFlZKHNP4I15P9udTzvXaG75o+aQS6U+7ibqcHpgwMvqk9rK1txGObG2WZeI0vTjQuS3KRfVoIDzT4deve4Oh+OEH8famHlCdutpwSKhdEFZuIeR2xLKmL86a5764OGvx3/Z53RbTrV+crXPWBv/dPOcoFtMmIL73yr4/l+Mon4yI1EcIoHrNa3XggfC+3t7e2QlD9GOrbul3IWcV4n0eZSBdmoTxPIRWsE3VqMIc3YItohnIBqqVSFFB8PpgmjA8vevIU7v8NftS04o7AxUx1xMJc+oJopbEiE9jLPCcEFIe79xx+fAUALNAnyfKAhU19gWbAlxak1Et4FBrurBYs6WdUtMJ6RB5nKXfKRIQb2fRNZq9uQcKnXAPvuryPMk81aF6kKoEb1lPmPVVv4kWMRkPJFBA8GkcmyFQIkuPQim3UENlBZkcRjAKPG+CY1cZCyCUnhNmJntRxmRSqT8OKrN1JGTYG4kux3AGjntAUiHV/sI4LHojNnLlEJ0xBXtxNHHFFDX1FEnK6iKzTIpozGtbKRlHUUJEzYV3LfTO8hnJzisSvG9EeBXOlNBlsDwKUPwlMlDuOXLRiJ7r3oM/rPvwEa8hMWc5IGL/8xsxICxU9cuoL1MR5WykjaMsSzMca2Cvik+T/V7OC0YC5whEPyNZYZtRTbsFuDWHRACRC1KovSKeQWUhb0Jih9i8H1FxuHsDIA/qzdJpoIf6ayiQCXUd4rxlh1FNva+J69Js3ZuwNGhMWBzQT/uc9OKbnAaVkHuJLQrH0UQSfny+gP129JTs9eGICUGIDYfKemXMowL5fVL/st9RPcY/0FZXx5AN07Rv8Hzy1ee/fP2HQHaZlOu+EY8fkzRdNxz8+vuKDPl3U2o6S2gGeTRQr8eVTDWpNkoJfnJ4+opKvL3piKX/UadpChVEWC2ZhJdm2Kkf5XVwpOVjX4xlmJP/3C95Sg9YVJDXM9ANMs9T1xKmPyyW0XQcJmnUh9z5Ms3GpCXfyP4TKvclmfpUsNS3eUEORlnbyPouJQEuymKyTbOE0h6LLkB+k2Zx/1jLTG9xwOt1saskgrjplAKTu+jxdKRhJIaGaqmQJS3YyhVXIzg91yK8joCbrjJT7yQ/iUHTK2xLBhQrfa9N0aAXA/V6kJi+K1Grir2pDJ/FOehlaZ6r19y71hguBzG7C4g3vrjWEp7Jt5yfIPaf0eie6L5rGf2azM7wQj5hkhEyM5pjdRbcCoNnL7784vDklPA4q72i4e5iAZO8D3mNh5dRv6/O0Z0giEO/R1FRUMq5Gfjf/+rFl69Q/a3g6rsE5yXJ1F6YTcIYNUhtXJPU4+fPSVrTE01fbgKF3ewXiCnzXqNCusUx+qq5o5eVplqsYNd9seGA3PQFKd6HvtimRNUDSt0hPQ1bhaq02pRxwhL/rEWVW7CCqFZri9JVB5FDEGDXtHZgGzk4fKekObH6gFldkeBscA4xoF4CxsVhWT0BSVEWWtFjeimbQGlXNV48IzzYlg6X0ARHa2ShdCtz6wB6W008LySB8sR575bT7UD89KeU91g8CfJp1zuo8KI7wXAaKxOeNjAhQs5qEPUYiqx6FBGy6SwXD0Tt5xOS7wfkDJxT64SpV00l9sf7UXql3vUm47K+zanUt6m6/nMsKNbN6TXg99zp7EKREjynArQSGpUaijcsA9aZPM+JkJVSPJRzhSoFFMeUJeoaCYeYAxBTT7XyNKdmJ7CS4iliKHPE8bs8oPqHJMI97zuybdXA23iNqpog7L5H9vQD0RKPUCeWyZDU1mOm2oD+fZefcfa5GWgLwtKIi52zJCJwj+0gcLrPqWCTanmAbRDYcwPwpu6ewbyBSoa9UVylUDoJrLneBTPVKCR+JmU75D1SysIZQZHxuTESlpwEbZwrdUwmSpiN80CcXumI7ZgX8EVXkhXWN+ZJPkqnWI0ju4dSaM6R3KOnsNDwqNXJaJZHvfz3eIQBC0FkY8TM2GrCEyn9q1EUywXTxpgBzw9enLx+cvD0D1ZZAmjtCVBke2C7YxRTiFVubJzqYt/pxiaZGUNyXPN6KSqKae/iOVX/dDlxstLYMMrhgPTeQTKM5RJlCz8QXpxf9q4+JxMSJRN+//Wp7L0Gjq3XrRakQyWlXYoJVS0lvMpukN7/qgtH4snsS7Jb2YhAWfiO91MQ8H6qfYU6n1CLkql0Z/7kiuCZItaMcLq7nAbUMVXx7jVUq2nwvU0Meulk5k2ugijBIRGi7HgaF9Eknnkn1lqwSVf18ujnncYRNOCCkwyhBgL3EvuK4NgsTca4vkii4pR0vlT+WNW9uZByoriYJhCOBeRmfrGNjOlFJiDNTdhJrxr5KMqKOQ/mq2mG2qWZmdM4jKUyJUOaSWFsTPyrCCazyONoTA4JN8GTC1zt+FFmRouRjPvkNKHtIrXODMsC8hmxmI5Vc4syYxeIA+UZiis4Yqp3SsKgNywX9F4e45bBikqFZ3vMcDhMjfYMdepAMyRB2hvzrrNAQPoK3jPtY2RQUMbd9MpXbepOpJmVV+IKEojEksE6omaAJgk6Fltoj+qxyTklzo6V3OnFBEl1UWPenYmDk5evT7Qoy2QjjMYGVTaorSzicqvkEJU+UTKoub7ZsQbyWBYkAXdtH4pR1LsgmBDvIRnGUGiCZrNu4otfNKmBBjk2ZCV98Uu8YOWFnk/5sUmy44svf0HPD/npl/S03eywW3/6/PDo6PUz7dr7+v1J+f7i88PKy+tXXx0Dqo+eKf/HMf0RAn5CA3lKvBNrEbky6pSkXKyuLaHOp4nTgTHwnxG7FevtgywLZx76+Rn6WA8GEaHQeKEOD2BHYPdONWyFsqnL5X6A8h10ngG22HC1X0UWkmjKpeeNHaOBpes4iPLTiyhJZP+lzEcsa9cwLdaioIBpOg5wUQ0xAT0p+Ur2gHlUeTgsTH6ZS1MrmNkBHJOATcFgsyAsiLjdKYG2kTF0aWWhnHBjc4tKXd1a6hsbotJdQ9voTh4tw4x4/IlybolKF5L4kkS5mkUkQjxP2dVrZHD+AIPxh1NtSvzAjsYPjNMPylz8AY7DD8rm++FIDn94lqbFD69SaWjYZTpVjEDMA/S62aGfR6AT6ZEptu9GDx6U9iCKdSH8mPu7oECjZewrC+hCAbogQBv0wwB0N1Ejv4KCe5qOJ9S9pPAiGI8dJtKV2CeoKM6wrzqmtTxaWkdtxzYk1jQ8Q53zec0seKyKr0knkrFsPHqCc0lwLoNwQurQBDjgkdpJV+9UjOEhfG8OnfPmTM+7DK7JziQBVBdrJG5oSgxnC0VmKPJLU6TjIE3wHokmOIOe9vdYSuFlZpNnKvmXiz0qg9VU6DOu+oDAuOAvgzdE08GbswuiCP/A1AnezJd5RKKBy3SXl+HuhN0c3a0TREz7s+GMKpgn02dTyDG2+VmLURRXLH3FLH1FKCsF8cCQhsT/MEuvlqobrn8VzpThPZ2sqYAkac9SaSd6x2ZpRMSUzyqMtHVRiWEzyj0ZxZ4Cb8fH0Qp3F7AdV3l8kpS1k2emJg+Gn4rSA6aPzb1Wudec+wt6KGcnsgeYkAauz6Et8zY32ftop4ETNdQS+al4qLTUvzYFrk2Ba1cU6PAQwMxo6GBHXaPKNV4qLDizrDwzrIyXa8v41y7jU3Of4Q9A4omq0EB9JrLb2H9Wsv91pW3F8TQB6kwbfnXzFbc/QtiC44BlvhIsmgvOHPhg9IFhWjvg8yW6xqq+c8BClTmdDxrYOAUJlsP5zDJo4a8KN/ji5XzGfFRBdZj6c5afc/Dq1KeZtStOgz7CW0lPvkq9Q4LEh4UO3VTEItR9Ps/d5Jd1Fa250fNeW0zURIDeB6csEMks2y4tPnI4L6UxICdRoSP+famWJfJYSkT6lXls/Wi23eOZmEzJWjCGKzvI5GlYo4znPnDzJpVIrGl7RNJmQAbuRIyneaGsYQPMe/DGN0FYZ60DAobM6HmHRRt79zXjGFUvmqUptaBBJh/XIJMlGuRTtYeLCeoC5qNyeH7601J+T1jI73GmkvK2coVupGnnOr9CJfk8/cxsUnOM56fY23NEFjCeQGNhiunnR3aK0UxlplmvokN46FHw4eqUIRd2tcqo9IBqT6DVVOjnKm3wckKGVYhdy1Ng/qsMe23Uspt45atlBxW4IgZUXpxarGJvi5pEJEjzyLeTNJbfBiXzRRdtj4ASHMRHfYF8dym1j4V065ojeHda51uDxBECjjpi5vG5Dy6fcLKp0I+wYnbaC+Mw845AtRazx5FhISzxq4fw2jvyy1EOiSAk+B6IlmysE9eF9NhFzAyvncpiA5X9THB5+u1SuSP6PQIzem1+IjFg2su/z4qywSbgqrqQ7PqqKW2OUxVQw/Yl7PfREdlXDobXT7CMhhL9tKC3er1kXkWW0+/JxXoElLfrNLp6N0ADoaIV4PpJoFTuZH7FQ/PJ2emt+FwvJk+o90SB2+sd1c816/Vb1O8p5HqjErbwRb8N5aLNGGfVzCypwtPXq6yKRxE9yVWAhOSqul9AW0YNWEZojo8BcCJLSgi3fFGgCe9byNBv69rt13EEg0yopxRfCWTCFmo9MrNS0GH6PI0vJWkmj2R0v4We+Syjmy7nHyidQGoH5Gey9FtLl7QUZfrt29e7DuH3HwS3jANhckB+NQ3WNyh72FxdtI2ipPEsy5VK5JB6QTZREzKTO8VvSzVLkha8Dk/aYhBdP66Eu9xlaNJ3V2FuDTkdndlT+yEWovU8AmfNoI2dMRv4s4U/27xXpoXkVrBxPm+sHd+xxzRNreAofCx2k7tdkBXVIl/7IOBR/wwpDeStAQVyupv1il9TUusY/gJTx/YKcQzy7Yi1Ljra0ropPWVVzAr5txUG2oXnyczJTzSM1tIA4SAsqt3kFaUAWpVthD01YsqYOgQE+v3GwGTz2jXHo0J7xQWkTBsPpRWsyPqKAZj2OiTL34gHqhlFkLNDagWWFTQBocK6AGyo6a1VghsO4N1VDsM5EWjKUuS8V5pOicbeCUWy4zNMJcdUjTfEud4Vrx/owKiqiMUOxBczY/b4WF/ojSTWP6Ic2pH85qzqCc8bKYcr3FwdY5+ClIcLRoeyUJr6kb0eYjNjeRBAq68OYZXsznk0du5PGLoZBYcT0fAjgVO+h7C0Hhg1FQ0TgITGJLsYdogyzan1FjXTaNVx+JurP+AQIo2txL0UhxM1wj2pMpqbpjnDfRPLdR3LV1Yjaslqpxg14AM9peAdD+fGYQiYT99UePpwQQiqGYDT9So+M1APK6UHVpSbwTrLkM1PEiIrZcjgtyVC9DSozILjqoCxUmUAQJDA1I266b8lN96JvmbwBtXcgcmcl0IMxgj5gX6ua5llSyFjv1QExGfHPz99/vrlwS/qRtxYOpgcLYgeMFTNQHVtyCqtLrHowVN5lxo0Z0BGeqkR07OK5v4y5bNyS45iSLeN3Kxa6jzmyDi6qAqJysjk1om5yzyanx8Lwrmj7ECenTnsRNUxO+84lycdMSxlI9cAXTXBUMZEodTYs+EI2pPAKcfJDulNaQNC5+wuwdOdctBJu7dOSjXSvrGloGWgcaAOoLVuePkBRbA6lrPrrS7AYdWGDOx8o0fq1dMxrtSwWzwM+nMbVR8HqPUyvFZLKlBYygxhQc5+bagYiY9+FvRIWg0LSgsGYWnLwWb5Sll4x1j2zcIr14pTgv3tTWeV70sF7rh1A9UJOuIQNKOUaVhHfRWZcI1KVYz32+lH7LlrmgCiWpxXa+vptDeCagNDkLWl3mG2gh79MmqQkrtHqtFdbcd1q2Mpy5ghvCrCJ1BX557ot8zE+tGBI+B+gj9HAdkFTljkhN7hrUD0VvztIx42EoYn/MAGE4uFiqExdplcVfFNjQUhv4rEn0DMcX1JECsDXLU8HvDRpRwKrD6H6aWLaqZRzQKeCBpXy/eB4vkHD9QaANVVvW/XnTJqMqBMmcazwm1oLssHHp/h4hXbETtX4jAvnkXXavwc54/fbu7Zcu7s9DhgbVPs2QIVKMh1zPdlOPHqnTmZyxEEXp1q7JvAQpyS5ix3Ag54mZZsLSaH40npzQBewsu8aqWLU8hG410CivYaC3aBF4paIbW4K0DDcD1gldRxI2hhNF6GQJ/1DtaoUU/ZndRTT9+PwRtYyvJ1taU1n98VanYf8/blfrm8ifX7PbWtB22fl2YHVmucjRBuG04/biPd7dtQfk4GptndCDR8J+DiCHiHjC4KdSPbu/zNCsj1mOV4xjIcF/FPi5GyzcwGOl7mgv1kN9ExW49mk7TwBmchZicJiMFZV81TTmrZpNZ5ea4C97M04Ux71yUgvuXIF25chmMxzhCDqM/AfF7PF7AAZDap8hbbYq7HzUdcyUbD6qk8K7DHceUWn7IWCXbez2nFE+95XFd7HHtBPCJIuF5ISy3sh9R7ILeQn3H+eQmPZJbVQI6oOkN8mNUvQno0/gUJPpJeaL7urjoD+YppV5q6XsQk/NrjDkaEldpgXfHW2Go7xgAee1dZvbPM6lK71e2Ob301FxwzTIj0KlFQlPLBvjBshS13Q6O6RZg6Ui6pgo++RhKxBTWtcQkg2kbBdZkw44SZsXS1mqVROCaDSVkvxx5pkrpvq7iWDZeqnxtBqmdodC0RvlxmIfyWB8NEAciW21PtAn1so2BTzdFgsMqsFlPRAQNkCicRomx+C+gLRKkqYxqzTWnLOts94bf32+6eMB6PZRvBsCYzKdQS/tcV8BjPb+ymz+dnO+cVaj/HYPri+dnmfHrr4bkrdub3kS4Iet2qFYVlWV7PUcs339x1QygbrGrbxZ6qabeDmv2hd9gVikNfRy/PBudnF+dqlVK/8T7QcgOobckMg97wqdJBnu48ecLz0tuobvi0PKuOrvUCPJAgsUsRfBGYa8U8c+c/l47Oq4MoQ2cQn3kbmxsKj2dee30DM+mZqtci2U3eUFPg7r1tU6a12cb2OyUfMF8dG7xyksXnVeHppMI/8eCWzZQQ3R6jB5d+U2Ptmgi1RPYuaj4DWpaNyjrbkFgXgws5g4+CO6uG4aQizChHXX/XUUoOV2Ul5HrgkNF8WaUFubTTM4Jwwhut84KJB62ArfHw7dfc5HUojHUsXFCVo2qV9W3SFOsPN+arbOG029Z6ZRS7OOXpqZNZWnd6XjNok2rNVGzCSIaKOueXABfNeN3YAwp1rI+oVESOMAas8lVaVpY8KUtmpqQRVIoo5Oh6BheXEnwWYK5TWzhjsNMCoHUAZggGrNtROdbbXOU1H73nMxi8S1OVkmMM/dcw6r0ao3ek1LPp1KpiJ1pNmy4tKRfiijmF2RKn4izxxfesDjQP1pcfz+Ele1IO7n5Xxevfa3Olw1u75/aYtj8X+apLOtjaXjy5q8rr07+4sgBb3XoFZim/q+1ICa6V8Wrtfnmu9/nBl5+zbdP0yYc6a/lt+tv21+nvur9Bf5v+Jv3d9Lfo75b/kP4+9Lc5ZYf+7vgkK6ga1ebqLR9nQZAMCDT+LcAgW6gFKGQv4UgIpwNS66HfAqzWtt8CtNaO325ym5RtT8Q8O3h6+ProxZeHbITRSJyRtKIG2v4m1XvYpsctgk+VNqjdh/S8Qemb1ImHD/2NZpN+t+nfjr++temv7zz029v0TmxI8u8cp9PPHjb9LUKpSb2k2s2H/iY9bcKkW6fq7Z11f51abO9s0e8WVd+k501VlcC0Nrb8HfrZbvnbhNdDf536u0HA1tstaheFqa0m9Y3aaW899IERJVANaqalUdj2dzb9bXRo298mumwQGJSibhB51tv03qRfUHjdp6YebquKJFYgcFr43aA21xltEhyQKSQ7qC2qukWjcs9StJezf3jJMkRvVMMNd6e4K8ez583NgzlhCiY6zlKyQoqZmieX9aDISNnVnR2lfTJm+I4ZNsvnD+Fablx26pYMlUnmxiBXHPXmVWOc97YxHH3a1t2qiHTVmjqIe3+vEv4353M/Q5uIBwi38C1FO7qgOtQ7X1KnmqKlHv/Gdl4fIn5uE8rjwZiyymthgjT5BAHpxOe3+DAd1w4tu4+N0SwrIDp49VaoD1fhbkIIlURvAS/3piAyjp1rdm+B0XXaxMx5/ZsaxVwXWfnadpybHq+/xzDvSYNkbOaTAiBF1pvpnJbOaZmc3MZDlf+oajdUbZKUufIcdRUafFJ3ZYu5yzEgFlG/Gezwan+O0P56sOXQj92kCbP+2TdoH4puohrs8QW6uYGxsanyuPXejPOcrsZAGyA2tp39MrgQ05wu83ExJDkQ+pQZD1TI22q7dqycxeVje+rsmM+bMTvkRZZeSJ6aGPY01ulo5xvNrFdO2tMQXkONGbKm07tyGCXHodqWwSmIJb5KvSAIDupOZZX0pNo2Kt2UHeSluj0IEa+mvoZWW/Ce0HlWkGdWJ7Dk0sKflUG7VA5b5zSuaF/5reG5rzire+7rQw9bMBCcxW1jDpead1CxkhW0s941+I7jD6QRFqC5OB+xB2F1Tb2607ildhof6X0ceqMxNzI4OzqLGq1zboUe8aCJA2ygitumvXlDgrwtwyNwYjXJdEzBguGv0gFOi5WSDinY7MxmMx3f3urplVRWLbJry9OM0Vgv6bxiRGExmMGimqGG0DUQ2sFmCWOJdRNOKkf6y3hOv8+XyOEMwk11BYhcw5m6C4CXgnthluGosQpP2JiDeuUrk0qouDH0Vfp12+u7mgiwIDbOzucYt4ceInZG+VXvCh9sY1t6GuGK5mCK0/ewp/CSp9MMti1uyMQ7fq0rD1gcEPbeCppcAESeZoRdiDWgV2MCohpzwxs8vcGHToacqK9EyK1/qPi/N8Vpro/BplIleHpxmsCb0wxe55qqLDBx0PN11L/eJfrYF2L5SY4U+vE1WW8qkVgMD8jv88hAMvEI1dzhiHj3bgWsOgdDuuHpL/hpk55+yU8b2gNV+4Ss/jILgMVsgi8N6XsZVHDadkOPrNrQ3XOcfX0AnY9bMJpYFhxPsChItH7zmpcVSW7TM9b0nTLcGSpGOZUToeaA354aVy03wCIcOi0PoFB3dabeaA0YzgL5IMo+oYa79IGl0YFqvg4al6GO1zbUoWIJaIL30e+pg0NY7aHqZ5BblHxfJ9dFWVKclQX8sjCHREj83lR8Vr3mtnTu4NfwbC/Ab5WpEqULdmmy+kJ9sBNP8Uj9Zvp3FOYvJ7vi/n0zXJUZlK+Ymzw0liwZH0WrxIA0Y8AqJe54TH9I8s4d/a3YZeUwka/vDI9vIjllgQ3EVtwS8aiS325Vs7PbsjtOyCtOL/SuJYyMeba7EVxOGC8GvThohk3YJQNYvnisfnZvG30f5CIGepOdObwD2nHCrmg6MTMtAHKE+DD5sNa92WyVBTiAXOdpo/huUubZVfL7CkCd8TZFaboadu3MbZ/GmUKc0uIh0cs5xUjda1KQ1LzAStAwnORzeOL2kQn15Owp9gmfUpfVFSokmswr+o8nj69Tga0Kxbx4WpzXDsmiIC5Rts4Ynft+SedQikhSZwY6G5MpSk7ebWXb/F0dMiFQlmqqwopQ39c1d6idevZAkobxlphMVWu356ply6rdWL7DujL6obQReyP82Sy2yFSivg6e16NHKgm4PAbfK/Lo3EznZpyblbn33DYZsTck2qzlgJVdlVddyr9x7uc4OvjyC5ZFMiEdqe/vReQwole+RZpeLpBnvjVQ7mJ49dXxi6dcGRcGUpETfEmSvKYMWtZ8/rd2yg8sivsyHJCXN+1DrTIktUtPjpX54eObMXwszdfuGm+FwtWiYEEzifnuT3tINOz1VDGtS8i3G2CRLyFCJ+kV/GgWcqSe+MJybI65tLszyY7Dx4TsXk0doCVb0Nwfx5fdVtfa+v0XhDM8ep4tL7TmR0NdLJGkWUQehqvTL7ED1lpfFfthDgSrZJIobMH0QJFCjbe4OdeDXVptwFtZFJdkHBhb4kyJFKarMYdQeg6/eSXgwIaNr6rrGx8aouW4PDBPTFikR9qlkDoo4plbQ/ENhAB3DYLsOg1J+rLIAA2cFtgd5UX1qvO6CjKuiQXcnHXkU/WxBzYolL1pUN0XO022QHRyHkc9ieDBThMb2Wp8F3pppgKg5g9FCAeb8W3o4NtBjM94DqFvf/KW6XyDm6B/8hZT7EyBPldGEx51podxCRLymmnw60GRPsPimNeq34j8W2DSDfg67j42euRjJiux3+ElNXWEu8kSmXk1AsLLAWqPg7polujKhFWsHEzwwTcC03WX2COz1E02AS6ZTfSR0PLojbqI2IldmSOiyo4gjZv0+fiq5w258QIqaxioneak9rB7cxhQyzoAjHOlNA4OiCqHqTLQsAanMjTgYHnhnMixoHhBqscOP81bc4kyX+o3mRujWq0EMLpllPvRJQCMAr6k9kslkmojrj1aHPdbxtrcMU6DzobBDazRn7ytEOHm27tNBY1VXsUqry3y8sJsoaHJeuruvYlhLbLDcyeWwBBgqPDXGHAMicaPE7hyGWOQH0dRVlGUNSTNDQUuSK/hoFGAz1g5eEm9IODsptGcrSQr64qoU6qKjlEGnXmOrrgCXb4mRs0L/hxeJhOsS5BgOjDnwL2a+w0ymltaWlVE2J4jxNRsIwfsyhz0dxFHcp6Ek/JGAWDPU6EMUpbbZln2nlGRc3bR9HiZqahOkjgXetTntn/oSyXQoN7H2dRBv0o4uoSDG05xJXd9bmw8Rdo1hJKq8gmSVOCCV6eXDJdUbuV8VOHGI/kmUBSh6Q51jahkq4ktXtYTTK86DoXM5nJtAlSDEMsoxRzwYA/tfqYMBdfzwjcPEL4yPSpNJ1Tb3+MC85umE9ODQZymGZdcUwVVaw3CGo1RiuFFQiCpnGTl1H1MS+K0kl91EWYPlfpIJHML33aMsfinbpXw+kX1Tk714PENDb455KhKfD+VeXFgPpOoWAQDxQxK4863udc/qlTeOlbYff3YEWX9Kt+Yso9FjS+Yh8KtHaOk8oWpnr5H/i4tf3x+z5mIqxHT6HTciWowUlfKL0UIJyykg5GxQL+cjrtUwKnMuyo1THcX4NeRvMIS09slosjKUBqrbHbKQi7NDsiXrZ3Z29fPcS3GUhmlb2JfJqNQkcXUpT65p65K1d9BoGQl5/V18XYtZy5braF25oXbb9CLj5swimDVntTnaIoPGBmaarV32fkx9GV9/WPoq1o19P0RNHFavgNNuMPV1ssbhQ9/cfDy+IiXhUEQdui+SFPeLM7XuXPgOE0kjiCGyQXuR2VS9dKxOuTWD2eBOI7x6VoR5lwgEyw/+FomfdMZiZqArzFnH/HDuz/98O4vxYd3f/vh1//xw7t/oAd6+Rt88+bXf/zh3Tt6+fcf3v03/vTNf//w7p/4M1u//g8ffv1OFVM59PdX9P6/qMCHd3/HSX+hYP7qw7s/oaf/wk/vBXIB4FcGzrs/IVCMDnup/DWcv/3w/h/4yzh/8eH9P314/3eB+PD+Vx/e/8mH93/14e//WOgyKPC3whR6/+Hvqchffnj/z5z9nrPf0yuV+HO3robJ+f8br3j4O270/3BtKvenaPE/OVCoOPXi/d84n+z5ZxARdw99KsvI6zsyzFtW6/iYlBZLOLyhmeSs5CKC11nCXBLfBCwB8JfEvXnBNs6HOCEks6yumsNHaeZlbtEpc9gaBMJkSgxxUVyN6hLG9+8DBsy9coKTPn1FRknuFXy5t0MoGd+JUgUT6q1zCVIxJpVUjM9kbPtZxi8J7BzmVMWG62rQYWTiX7JnxpfX3AvzWdITJcbRMHlRVG7aLtQVinMDYbcJ6Hsspvlsfq3+fsH3Ir5lEtf0B9+sD8GRvKC2fISMu2SMh/uId1hQRwh+6K9miISUlrpF+WlKPqtQN2zDQitkN00vcGu7PitvvpyuV8r5jmZ1xAKKlphZaG3qNq9jK1YnD9MazrzrD9+bjHKkbVyLd+7g+1T1OR+Hv98NsI4DVzSXhX2c/GjM93zzbXYFdTGMPTVFQJFvv0mzC/V9Pjjk3gIgslWLprVUrQ3cZB+97ot1fUShyGYV8xRNhldhVIiBLHoj0iaTaM1+nsrHgMhilGJx6vir01eIukn+iMIuZdU0FzYw9Pg+BCx8872S73J8mOXG13sYfv/0qy+xOE2diAYzT30cWke9bupziw+kzCxe+EYDbzns4cZxTROqn17smuiYxBXexPi8O0JxR5jkVxK3Pf/kbaa/KnvzLbXiBiKplSC9qHMY+Yqj9IcAhJtNAwYJv7I2TS4S2HCcUr1/FQEiE2dDJRW/yviqL7sshlcOmvEE6cN5wHEZ8HvNbp3VLuOn24gONsVYXcOCj3QCBF8Apwrczqn4grqG5MozInhvRAjbBYS3wn4bbVeY/pbmBXvijxG7+s8QQwTAFveFYSlONy/Yzyd54fMROV1EFV4P5iJqZdhwBU8B/f2vKPmkGaCiVHw41OlRjV55NLoS9w/wnvicTXUyIFKCWa+pW96o4re71CC441QWdpjtwqtamFRxJRV6qAe40//GCsIfUVkfY63lCguaRNRlWxuROM1Vc/2ExjEfG/pWLQgLnjbCU+rv49xQKylO0xtXOeKW/Fx9HnGXYyCABX+XfgJilRy3+T+uvO2ytjXSltEYREkYx/jILm+WsjKO5R5Chm6Ee5kYtssY7BFy7mrjQmk55TJp1bNY+ELOcIMFbBF56aytysuAv3wBnuZPPNbQWyT3iiz+A8mXyNAbScaQ3lh90yuZ4ID/uRyE07jgbTta17Im5uh8KWNJjMbFCPN6JBNPbw72skCtn2iZZ1ZRTKmRXffTywP373sjIDeCINM6ib/pOi8vuPxj8S1rVo7qjoLhZEoykcb0q8GAt0BROkKwIeSmo0tjpcdK0IsGUproJQt4GFVR/XYlTndpV9HtOCPzP5cBcZCH7aSajrwM8dp+YZaJdmeKujiq1doSsPI1g+7WRlAw87ggM+6GZweFSi1sS+hGcE1/Tkpgm1d+A6yAeGGRdj0GyqYVTRCvp4a0NwrIf8+epn2J78nUq2ti+ksf/AGPA5hzHsEPuvwVHh9SSq2Y+ffODU5nIApJ3bgYnFvkeF02xM1S89tnkKr3WaiViHoAZsYNmvV6ZSkp5M/TWOXG76zdvj1UAwdhyYtuZ7xpIMjJIii82mte8dHxZkq/+ZYQ78txWjNXdyLuS+gGpNayz4n8AaUhy3yCiTDCDrif41PFEDPZDDctErK5x/U4rlSv1CCnkQglT/VtlwsFl10YmmqmTWnIpnkxHT8l3qgIIK5aucy+41xY3Vm4n1XtEjVfvrFtVi47Vp/yUeEz9RzoM5x7OlLGUPQ3o8rgh9kkADjuWljd2hRzC2SV+ck6QTOHhT03R53vrfSsOuC7h71P0QUIxZKVfiK5Kn8SBZm4NYQs0nAYEkNbaxlH8dncYgyBsfqskt5bzJ+e4W+deDWPBAfNgrxBsmPak/3GOFXfoVPvdWqUi0ve8F9aV6WeWWFe6TjgvVuDk/YQbzccTXmVGUOWkYU8g4dl5u/9++Bl2MJDs5TsZrtjxKUmsO28pQjfYI9MoaKNHFaygcfLuXghVq7LT6v9fwth8VW7nQAA")))
AV_DIR = os.path.join(ROOT, "avatar"); os.makedirs(AV_DIR, exist_ok=True)

# the avatar (pixiv's VRM1 Constraint Twist Sample, VRM Public License 1.0), kept on Drive after the first download
AV_B64 = os.path.join(AV_DIR, "avatar.b64.txt")
if not os.path.exists(AV_B64):
    url = "https://raw.githubusercontent.com/pixiv/three-vrm/release/packages/three-vrm/examples/models/VRM1_Constraint_Twist_Sample.vrm"
    vrm_bytes = urllib.request.urlopen(url, timeout=120).read()
    assert hashlib.md5(vrm_bytes).hexdigest() == "b36a6fbb65316d5684de896dedee4f26", "unexpected avatar file"
    open(AV_B64, "w").write(base64.b64encode(vrm_bytes).decode())
    print("avatar downloaded")

def first_file(*names):
    for n in names:
        f = os.path.join(AV_DIR, n)
        if os.path.exists(f):
            return f
    return None
EMPTY = os.path.join(LIVE_DIR, "empty.json"); open(EMPTY, "w").write("[]")
FILES = {"/": os.path.join(LIVE_DIR, "app.html"), "/app.html": os.path.join(LIVE_DIR, "app.html"),     # the BahuSanket front end
         "/viewer.html": os.path.join(LIVE_DIR, "viewer.html"), "/avatar.b64.txt": AV_B64,          # the research viewer, same data
         "/clips_paragraph.json": first_file("clips_paragraph_19.json"),                      # notebook 19's six demo paragraphs
         "/clips_model.json": first_file("clips_model_18.json", "clips_model.json"),          # test sentences, model + real
         "/clips.json": first_file("clips.json") or EMPTY}                                    # real iSign clips (optional)
for k, v in FILES.items():
    print(f"  {k:<22} {v if v else '(not on Drive: skipped)'}")
TYPES = {".html": "text/html; charset=utf-8", ".json": "application/json", ".txt": "text/plain"}

class Handler(http.server.BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    def log_message(self, *a):
        pass
    def _send(self, code_, body, ctype):
        self.send_response(code_); self.send_header("Content-Type", ctype); self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store"); self.end_headers(); self.wfile.write(body)
    def _json(self, obj, code_=200):
        self._send(code_, json.dumps(obj, ensure_ascii=False, separators=(",", ":")).encode("utf-8"), "application/json")
    def do_GET(self):
        p = self.path.split("?")[0]
        if p == "/api/health":
            return self._json({"ok": True, "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu", "langs": list(LANGS)})
        f = FILES.get(p)
        if not f:
            return self._send(404, b"not found", "text/plain")
        self.send_response(200); self.send_header("Content-Type", TYPES.get(os.path.splitext(f)[1], "application/octet-stream"))
        self.send_header("Content-Length", str(os.path.getsize(f))); self.end_headers()
        with open(f, "rb") as fh:
            while True:
                b = fh.read(1 << 20)
                if not b:
                    break
                self.wfile.write(b)
    def do_POST(self):
        if self.path.split("?")[0] != "/api/generate":
            return self._send(404, b"not found", "text/plain")
        try:
            req = json.loads(self.rfile.read(int(self.headers.get("Content-Length", 0))) or b"{}")
            self._json(make_paragraph(req.get("text", ""), req.get("lang", "auto")))
        except Exception as e:
            print("request failed:", repr(e), flush=True)
            torch.cuda.empty_cache()
            self._json({"ok": False, "error": f"{type(e).__name__}: {e}"})

class Server(socketserver.ThreadingMixIn, http.server.HTTPServer):
    daemon_threads = True; allow_reuse_address = True
PORT = int(os.environ.get("BAHU_PORT", 8000))
try:
    SERVER.shutdown(); SERVER.server_close()          # re-running this cell restarts the server
except NameError:
    pass
SERVER = Server(("0.0.0.0", PORT), Handler)
threading.Thread(target=SERVER.serve_forever, daemon=True).start()
print(f"server running on port {PORT}")
if not os.environ.get("BAHU_ROOT"):
    from google.colab.output import eval_js, serve_kernel_port_as_window
    LINK = eval_js(f"google.colab.kernel.proxyPort({PORT})")
    print("\nOPEN THIS LINK (keep this Colab tab open while presenting):\n  " + LINK +
          "\n(the research viewer with all the switches is at the same link + viewer.html)")
    serve_kernel_port_as_window(PORT, anchor_text="Open BahuSanket (live)")

### Optional: a public link (only if the link above does not open)

Some browsers block the Colab link (for example strict cookie settings). This cell makes a temporary public `trycloudflare.com` link
instead, with Cloudflare's free quick tunnel (no account needed). Anyone with that link can use the demo while this notebook runs,
so share it with no one and stop it (Runtime → Interrupt) after the presentation. Set `PUBLIC_LINK = True` to use it.

In [ ]:
# 12. (optional) public link through a Cloudflare quick tunnel
PUBLIC_LINK = False
if PUBLIC_LINK:
    if not os.path.exists("/content/cloudflared"):
        subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        os.chmod("/content/cloudflared", 0o755)
    TUNNEL = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in TUNNEL.stdout:
        m_ = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m_:
            print("PUBLIC LINK (temporary):", m_.group(0)); break
    threading.Thread(target=lambda: [None for _ in TUNNEL.stdout], daemon=True).start()   # keep reading its log so it never blocks
else:
    print("not used (set PUBLIC_LINK = True only if the Colab link does not open)")